# 📖 Isolation Test: Retrieval-Augmented Generation (RAG)

This notebook focuses strictly on the Vector Search capabilities. We will load the pedagogy guidelines, chunk them, embed them locally using `sentence-transformers`, search them using `FAISS`, and verify the context extraction.

In [ ]:
import os
from dotenv import load_dotenv
from rag import PedagogyRAG

load_dotenv(override=True)
print('Environment constraints initialized.')

In [ ]:
# 1. Initialize the FAISS Vector Database
# This automatically loads 'pedagogy_guidelines.md', chunks it, and vectorizes it.
rag_system = PedagogyRAG()
print('\nFAISS Database constructed from pedagogical guidelines.')

In [ ]:
# 2. Test Simulating a Vector Search Query
# Usually, the GapDetector formulates this query inside the Agent.
test_query = "Bloom's Remember level questions and addressing gaps like: High Failure Rate"

print("Querying FAISS Database for:\n", test_query)
retrieved_context = rag_system.retrieve_guidelines(test_query)

print("--------------------------------------------------")
print("\n🔍 RETRIEVED CONTEXT (Top K chunks):\n")
print(retrieved_context)
print("--------------------------------------------------")

### Standard LLM Generation (Without LangGraph Agents)
We can optionally pipe this directly into an LLM just to verify if the LLM understands it!

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0.2, groq_api_key=api_key)

prompt = PromptTemplate.from_template(
    "You are an expert educational consultant. Use ONLY the extracted RAG context below to explain how to fix a 'Remember' level question with high failure rate.\n\n"
    "Context:\n{context}\n\n"
    "Explanation:"
)

chain = prompt | llm
response = chain.invoke({'context': retrieved_context})

print("🤖 LLM Insight Using RAG Context:\n")
print(response.content)